In [1]:
import numpy as np
import torch 
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings("ignore")

In [2]:
from data import get_physics_dataset, UnifiedDatasetWrapper


def form_dataset(train: bool = False) -> torch.utils.data.Dataset:
    data_filepath = 'data_demo/caloGAN_case11_5D_120K.npz'
    return UnifiedDatasetWrapper(get_physics_dataset(data_filepath, train=train))

data = form_dataset(train=True)
val_data = form_dataset(train=False)
dl = torch.utils.data.DataLoader(data, batch_size=32)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=32)

In [3]:
for i in val_loader:
    energy = i[0]
    point = i[1][0]
    momentum = i[1][1]
    print(i[0].shape)
    print(i[1][0].shape)
    print(i[1][1].shape)
    break

torch.Size([32, 1, 30, 30])
torch.Size([32, 2])
torch.Size([32, 3])


In [4]:
from diffusion import DiffusionModel

dm = DiffusionModel(100, 16)
dm.load_state_dict(torch.load('checkpoints/t100.pt', map_location=torch.device('cpu')))
dm.eval();

In [5]:
# dm.eval()
# with torch.no_grad():
#     sample = dm.sample_single(momentum[10], point[10], plot=True)

In [6]:
samples = dm.sample(momentum, point)

  0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
def compare(idx=0, truncate=None):
    real, sampled = energy[idx].detach().numpy(), samples[idx].detach().numpy()
    vmin, vmax = min(real.min(), sampled.min()), max(real.max(), sampled.max())

    fig, axs = plt.subplots(1, 2, layout='constrained', sharex=True, sharey=True)
    sum1, sum2 = energy[idx].sum(), samples[idx].sum()
    im1 = axs[0].imshow(np.transpose(real, (1, 2, 0)), cmap='inferno', vmin=vmin, vmax=vmax)
    axs[0].set(title=f'real {sum1:.2f}')
    if truncate:
        sampled[sampled < truncate] = 0
    im2 = axs[1].imshow(np.transpose(sampled, (1, 2, 0)), cmap='inferno', vmin=vmin, vmax=vmax)
    axs[1].set(title=f'generated {sum2:.2f}')
    fig.colorbar(im2, fraction=0.05)
    plt.show()

In [8]:
# for i in range(32):
#     compare(i)

In [10]:
from evaluate import calc_stats

calc_stats(dm, val_loader, 3)

  0%|          | 0/1875 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

({'Longitudual Cluster Asymmetry': array([-0.67534436, -0.88781218, -0.98421165, -0.95052322, -0.94136429,
         -0.83156065, -0.95389378, -0.95789928, -0.95199547, -0.95010727,
         -0.19453804, -0.86417456, -0.8992491 , -0.96550398, -0.17818409,
         -0.94032613, -0.85540487, -0.96765892, -0.56063761, -0.991526  ,
         -0.96102619, -0.92120573, -0.14374785, -0.94373465, -0.90390199,
         -0.93539907, -0.97508456, -0.92967442, -0.97280277, -0.98378941,
         -0.47724701, -0.68731293]),
  'Transverse Cluster Asymmetry': array([ 0.92651031, -0.55578985, -0.12058268,  0.61006037,  0.24719393,
          0.50432029,  0.7072894 , -0.28761982, -0.78481478, -0.2954034 ,
         -0.25397843, -0.36511943, -0.57603827, -0.75932913,  0.82558096,
         -0.78326315, -0.81453229, -0.68939294, -0.11092626,  0.52971124,
         -0.68528106,  0.81181945, -0.47991901,  0.65915113,  0.89038034,
         -0.940919  ,  0.22706968,  0.53589235, -0.6541442 , -0.45216738,
          